# Comparing the selected training data

After the 8 methods select their training data, I compare what each group contains and how many pairs they share. Since all methods select 256 pairs, the point of this comparison is to examine the differences in the selected examples.

I also inspect the response review table used during training preparation, as the safe counterpart's generated response becomes the reference answer for LoRA fine-tuning.

- [The eight selected data groups](#The-eight-selected-data-groups)
- [What did each method select?](#What-did-each-method-select?)
- [How many pairs do the methods share?](#How-many-pairs-do-the-methods-share?)
- [Reviewing the reference responses](#Reviewing-the-reference-responses)


## The eight selected data groups

The function `selection()` reads the chosen pairs for each method, and the dictionary variable `selections` keeps these as separate tables so their counts and overlap can be compared.


In [ ]:
# compare the chosen pairs before interpreting fine-tuning results

from pathlib import Path
import sys
import json
import pandas as pd

root = next(path for path in [Path.cwd(), *Path.cwd().parents] if (path / "scripts/curation/configs/config_qwen7b.json").exists())
sys.path.insert(0, str(root))

# use the same Qwen2.5-7B configuration across all selection methods
from scripts.curation.selection.extract_results import configuration, selection, validation
config, output = configuration()
pd.set_option("display.max_colwidth", None)

selections = {path.name: selection(path.name) for path in sorted((output / "selections").iterdir()) if (path / "pairs.jsonl").exists()}

## What did each method select?

The table variable `summary` shows how many pairs and distinct injection methods each group contains, as well as how many selected pairs caused the base model to produce the requested injected answer.

The probes and Jlens use the same pairwise selection score, while random selection, text diversity and model failures select pairs through their respective rules.


In [ ]:
# count selected pairs separately from the number of injection methods they cover
summary = pd.DataFrame([{"method": method, "pairs": len(rows), "injection_methods": rows.attack_method.nunique(), "witness_redirections": int(rows.witness_redirection.sum())} for method, rows in selections.items()])
display(summary)

## How many pairs do the methods share?

I compare the pair IDs between every two selection methods and count how many appear in both groups, using the table variable `overlap` to show these counts.

A higher count means the two methods selected more of the same data, while each diagonal entry compares a method with itself and therefore gives its full group size.


In [ ]:
# a shared pair appears in both training groups even when their selection scores differ
overlap = pd.DataFrame(index=selections, columns=selections, dtype=int)
for first, rows in selections.items():
    for second, other in selections.items():
        overlap.loc[first, second] = len(set(rows.pair_id) & set(other.pair_id))

display(overlap.astype(int))

## Reviewing the reference responses

The table variable `review` contains the selected pairs and their response-review fields, with the column `safe_task_correct` recording whether the safe response completes the task and `injection_followed` recording whether the injection response follows the inserted instruction.

These fields were introduced for the earlier hidden state separation experiment. For the final reference-response training, I used the original model-generated safe answers without systematically verifying the correctness of every task response.


In [ ]:
# inspect response quality separately from the scores used to select pairs
review = pd.read_csv(output / "training_review.csv", keep_default_na=False)
print("Distinct pairs reviewed or awaiting review:", len(review))
display(review.head(3))